# Préparation du jeu de données version 2

Ce notebook construit le jeu commun du modèle **bon état / endommagé**, sans modifier `data/sources/`.
Il crée une copie préparée dans `data/v2/` :

```
data/v2/
    train/  bon_etat/  endommage/
    val/    bon_etat/  endommage/
    test/   bon_etat/  endommage/
```

Sources :
- **casting** : photos originales de roues de pompe moulées (`casting_512x512`);
- **pneus** : photos de pneus prises au téléphone (Mendeley Data);
- **roulements** : roulements à billes photographiés sur banc (Roboflow « bearing-defect-qc »);
- **ecrous** et **vis** : écrous et vis du jeu MVTec AD (rayures, déformations, filetages abîmés);
- **joints** : joints toriques intacts ou cassés (Roboflow « o-ring data »);
- **acier** : surfaces d'acier prêtes pour la peinture ou à retraiter : rouille, calamine, rayures (SteelBlastQC).

Le jeu Roboflow « Rust Detection » a été examiné puis **exclu** : les photos sans corrosion sont des vis neuves
en studio et les photos corrodées des tuyaux en extérieur. Le fond suffisait à prédire la classe.

Étapes :
1. choisir les dossiers sources et leur classe;
2. retirer les doublons exacts (empreinte MD5);
3. découper en entraînement, validation et test, **par groupe** quand des images se ressemblent trop;
4. copier les images en les réduisant à 512 pixels au maximum;
5. enregistrer un manifeste (`resultats/v2_manifeste.csv`) qui indique d'où vient chaque image.

Cliquez sur **Run All**. La préparation prend quelques minutes.

In [ ]:
import csv
import hashlib
import random
import re
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageOps

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOURCES = RACINE / "data" / "sources"
SORTIE = RACINE / "data" / "v2"
MANIFESTE = RACINE / "resultats" / "v2_manifeste.csv"
SEANCES_ROULEMENTS = RACINE / "data" / "roulements_seances.csv"

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
TAILLE_MAX = 512
PARTS = {"train": 0.70, "val": 0.15, "test": 0.15}
GRAINE = 42

plt.rcParams["figure.dpi"] = 110

## 1. Dossiers sources retenus

**Casting** : seules les photos originales (`casting_512x512`) sont utilisées. Le dossier `casting_data`
contient des versions augmentées des mêmes pièces, qui risqueraient de se retrouver à la fois en entraînement et en test.

**Pneus** : les deux dossiers du jeu Mendeley.

**Roulements** : les photos sont regroupées en **séances** (rafales de photos du même roulement), et chaque séance
entière va dans une seule partie, selon `data/roulements_seances.csv`. La graisse visible est exclue (ce n'est pas
un défaut) et les rayures (une seule séance) sont gardées pour un test externe.

**MVTec AD (écrous et vis)** : les dossiers `ground_truth` contiennent des **masques** (images noir et blanc qui
indiquent où est le défaut), pas des photos : ils sont exclus. Pour les écrous, le défaut `flip` (écrou posé à
l'envers) est exclu, car la pièce n'est pas abîmée.

**Joints** : les images sont des **images extraites de vidéos** (`breakage1_mp4-0`, `breakage1_mp4-1`…), et la classe
de chaque image est donnée dans `_classes.csv`. Deux images voisines d'une même vidéo sont presque identiques :
chaque **vidéo** forme donc un groupe, qui va entièrement dans une seule partie. Le découpage proposé par Roboflow
(`train`, `valid`, `test`) n'est pas utilisé, car il mélange les images d'une même vidéo.

**Rouille (Rust Detection)** : exclue. Les photos sans corrosion sont des vis neuves en studio et les photos
corrodées des tuyaux en extérieur : le fond suffit à prédire la classe.

**Acier (SteelBlastQC)** : chaque grande photo de plaque a été découpée en tuiles (`slice035_000` à `slice035_015`).
Les tuiles d'une même photo se ressemblent et peuvent appartenir aux deux classes : chaque photo (`slice035`)
forme donc un groupe, qui va entièrement dans une seule partie, **quelle que soit la classe de ses tuiles**.
Le découpage train / test fourni avec le jeu n'est pas utilisé, car il sépare les tuiles d'une même photo.

In [ ]:
PNEUS = "pneus/Digital images of defective and good condition tyres"
SELECTION = [
    # (nom de la source, dossier dans data/sources, classe)
    ("casting", "casting/casting_512x512/casting_512x512/ok_front", "bon_etat"),
    ("casting", "casting/casting_512x512/casting_512x512/def_front", "endommage"),
    ("pneus", f"{PNEUS}/good", "bon_etat"),
    ("pneus", f"{PNEUS}/defective", "endommage"),
    # MVTec AD : écrous (le défaut « flip » est exclu)
    ("ecrous", "mvtec_ecrous/metal_nut/train/good", "bon_etat"),
    ("ecrous", "mvtec_ecrous/metal_nut/test/good", "bon_etat"),
    ("ecrous", "mvtec_ecrous/metal_nut/test/scratch", "endommage"),
    ("ecrous", "mvtec_ecrous/metal_nut/test/bent", "endommage"),
    ("ecrous", "mvtec_ecrous/metal_nut/test/color", "endommage"),
    # MVTec AD : vis
    ("vis", "mvtec_vis/screw/train/good", "bon_etat"),
    ("vis", "mvtec_vis/screw/test/good", "bon_etat"),
    ("vis", "mvtec_vis/screw/test/scratch_head", "endommage"),
    ("vis", "mvtec_vis/screw/test/scratch_neck", "endommage"),
    ("vis", "mvtec_vis/screw/test/thread_side", "endommage"),
    ("vis", "mvtec_vis/screw/test/thread_top", "endommage"),
    ("vis", "mvtec_vis/screw/test/manipulated_front", "endommage"),
    # Rouille : exclue après inspection (bon état = vis en studio, endommagé = tuyaux en extérieur)
    # *[("rouille", f"rouille/{p}/Corrosion", "endommage") for p in ("train", "valid", "test")],
    # *[("rouille", f"rouille/{p}/no-corrosion", "bon_etat") for p in ("train", "valid", "test")],
    # Acier (SteelBlastQC) : surfaces prêtes pour la peinture ou à retraiter (rouille, calamine, rayures)
    *[("acier", f"acier/SteelBlastQC/{p}/good", "bon_etat") for p in ("train", "test")],
    *[("acier", f"acier/SteelBlastQC/{p}/not-good", "endommage") for p in ("train", "test")],
]
# Sources dont un même groupe peut contenir les deux classes : découpage commun aux deux classes
GROUPES_MULTICLASSES = {"acier"}

def groupe_roboflow(nom):
    """Photo d'origine d'un fichier Roboflow; pour une image tirée d'une vidéo, la vidéo."""
    base = nom.split(".rf.")[0]
    video = re.match(r"(.+_mp4)-\d+", base)
    return video.group(1) if video else base

def groupe_acier(nom):
    """Photo d'origine d'une tuile SteelBlastQC (slice035_011.png → slice035)."""
    tuile = re.match(r"(slice\d+)_\d+", nom, re.IGNORECASE)
    return tuile.group(1).lower() if tuile else Path(nom).stem

def ligne(source, classe, fichier, groupe="", partie=""):
    return {"source": source, "classe": classe, "groupe": groupe, "partie_imposee": partie,
            "origine": str(fichier.relative_to(SOURCES).as_posix())}

lignes = []
for source, dossier, classe in SELECTION:
    chemin = SOURCES / dossier
    if not chemin.is_dir():
        raise SystemExit(f"Dossier introuvable : {chemin}")
    for fichier in sorted(chemin.iterdir()):
        if fichier.suffix.lower() in EXTENSIONS:
            groupe = ""
            if source == "rouille":
                groupe = f"rouille:{groupe_roboflow(fichier.name)}"
            elif source == "acier":
                groupe = f"acier:{groupe_acier(fichier.name)}"
            lignes.append(ligne(source, classe, fichier, groupe))

# Joints : la classe de chaque image est dans le fichier _classes.csv de chaque dossier
CLASSES_JOINTS = {"good": "bon_etat", "breakage": "endommage"}
desaccords = 0
for partie_roboflow in ("train", "valid", "test"):
    dossier = SOURCES / "joints" / partie_roboflow
    tableau = pd.read_csv(dossier / "_classes.csv")
    tableau.columns = [c.strip() for c in tableau.columns]
    for _, r in tableau.iterrows():
        actives = [c for c in CLASSES_JOINTS if c in tableau.columns and r[c] == 1]
        if len(actives) != 1:
            continue  # aucune classe ou deux classes : image ignorée
        classe = CLASSES_JOINTS[actives[0]]
        if not r["filename"].startswith(actives[0]):
            desaccords += 1  # le nom du fichier contredit l'étiquette
        lignes.append(ligne("joints", classe, dossier / r["filename"], f"joints:{groupe_roboflow(r['filename'])}"))
print(f"Joints : images dont le nom contredit l'étiquette : {desaccords}")

# Roulements : classe et partie imposées par séance
DOSSIER_ROULEMENTS = SOURCES / "roulements"
with open(SEANCES_ROULEMENTS, encoding="utf-8") as f:
    seances = list(csv.DictReader(f))
manquantes = 0
for s in seances:
    if s["partie"] not in ("train", "val", "test"):
        continue  # graisse exclue, rayures réservées au test externe
    chemin = DOSSIER_ROULEMENTS / s["image"]
    if not chemin.exists():
        manquantes += 1
        continue
    lignes.append(ligne("roulements", s["classe"], chemin, f"roulements:{s['seance']}", s["partie"]))
if manquantes:
    raise SystemExit(f"{manquantes} image(s) de roulements introuvable(s) dans {DOSSIER_ROULEMENTS}.\n"
                     "Vérifiez que mechanical_v2_bearing.zip est extrait dans data/sources/roulements/.")

images = pd.DataFrame(lignes)

# Acier : familles de noms de fichiers par classe (une famille présente dans une seule classe serait suspecte)
acier = images[images["source"] == "acier"]
if len(acier):
    famille = acier["origine"].map(lambda o: re.sub(r"[\d_() ]+$", "", Path(o).stem))
    print("Acier : familles de noms de fichiers par classe")
    print(pd.crosstab(famille, acier["classe"]).to_string(), "\n")
    classes_par_photo = acier.groupby("groupe")["classe"].nunique()
    print(f"Acier : {len(classes_par_photo)} photos d'origine, dont {int((classes_par_photo > 1).sum())} "
          "avec des tuiles dans les deux classes\n")

images.groupby(["source", "classe"]).size().unstack().rename_axis(None, axis=1)

## 2. Retrait des doublons exacts

In [ ]:
def empreinte(chemin):
    return hashlib.md5((SOURCES / chemin).read_bytes()).hexdigest()

images["md5"] = [empreinte(c) for c in images["origine"]]
doublons = images[images.duplicated("md5", keep="first")]
print(f"Doublons exacts trouvés : {len(doublons)}")
if len(doublons):
    display(doublons.groupby(["source", "classe"]).size().rename("doublons").to_frame())

# Une même image dans deux classes différentes serait une erreur d'étiquette : on la retire complètement
conflits = images.groupby("md5")["classe"].nunique()
conflits = set(conflits[conflits > 1].index)
print(f"Images présentes dans les deux classes (retirées) : {len(conflits)}")

images = images[~images["md5"].isin(conflits)].drop_duplicates("md5").reset_index(drop=True)
print(f"Images conservées : {len(images)}")

## 3. Découpage entraînement / validation / test

- **Roulements** : la partie de chaque séance est imposée par `data/roulements_seances.csv`.
- **Joints et acier** : découpage **par groupe** (vidéo ou photo d'origine). Pour les joints, chaque classe est
  découpée séparément; pour l'acier, les deux classes ensemble, puisqu'une même photo contient des tuiles des deux classes.
  Les groupes sont mélangés, puis attribués au test jusqu'à environ 15 % des images, à la validation jusqu'à
  environ 15 %, et le reste à l'entraînement. S'il y a moins de 4 groupes, la validation est sautée pour garder
  au moins deux groupes en entraînement.
- **Casting, pneus, écrous et vis** : découpage aléatoire image par image, séparément pour chaque classe (70 % / 15 % / 15 %).

In [ ]:
rng = random.Random(GRAINE)
images["partie"] = images["partie_imposee"]

# Joints et acier : groupes entiers
a_grouper = (images["partie_imposee"] == "") & (images["groupe"] != "")
images["cle_decoupage"] = images["classe"].where(~images["source"].isin(GROUPES_MULTICLASSES), "toutes")
for (source, cle), lot in images[a_grouper].groupby(["source", "cle_decoupage"]):
    tailles = lot.groupby("groupe").size()
    groupes = sorted(tailles.index)
    rng.shuffle(groupes)
    cibles = [("test", PARTS["test"])] + ([("val", PARTS["val"])] if len(groupes) >= 4 else [])
    for partie, part in cibles:
        compte = 0
        while len(groupes) > 2 and compte < part * len(lot):
            g = groupes[-1]
            if compte and compte + tailles[g] > 1.3 * part * len(lot):
                break  # ce groupe ferait trop dépasser la cible
            groupes.pop()
            images.loc[lot.index[lot["groupe"] == g], "partie"] = partie
            compte += tailles[g]
    images.loc[lot.index[lot["groupe"].isin(groupes)], "partie"] = "train"

# Autres sources : image par image
for (source, classe), lot in images[images["partie"] == ""].groupby(["source", "classe"]):
    indices = list(lot.index)
    rng.shuffle(indices)
    n = len(indices)
    n_train = round(n * PARTS["train"])
    n_val = round(n * PARTS["val"])
    images.loc[indices[:n_train], "partie"] = "train"
    images.loc[indices[n_train:n_train + n_val], "partie"] = "val"
    images.loc[indices[n_train + n_val:], "partie"] = "test"

repartition = images.pivot_table(index=["source", "classe"], columns="partie", values="md5",
                                 aggfunc="count").reindex(columns=["train", "val", "test"]).fillna(0)
repartition.loc[("Total", ""), :] = repartition.sum()
repartition.astype(int)

In [ ]:
# Vérification : aucun groupe (séance, vidéo ou photo d'origine) ne doit être partagé entre deux parties
parties_par_groupe = images[images["groupe"] != ""].groupby("groupe")["partie"].nunique()
print("Groupes présents dans plusieurs parties :", int((parties_par_groupe > 1).sum()))
groupes = images[images["groupe"] != ""].groupby(["source", "classe", "partie"])["groupe"].nunique()
print("\nNombre de groupes par partie :")
print(groupes.unstack().reindex(columns=["train", "val", "test"]).fillna(0).astype(int).to_string())

fig, ax = plt.subplots(figsize=(8, 3.8))
tableau = images.groupby(["partie", "classe"]).size().unstack().loc[["train", "val", "test"]]
tableau.plot.bar(ax=ax, color=["#2f9e44", "#c92a2a"], rot=0)
for conteneur in ax.containers:
    ax.bar_label(conteneur, fontsize=8, padding=2)
ax.set_xlabel("")
ax.set_ylabel("Nombre d'images")
ax.set_title("Jeu version 2 : images par partie et par classe")
ax.legend(title="")
plt.tight_layout()
plt.show()

## 4. Copie des images préparées

Chaque image est orientée correctement, convertie en couleur (RGB) et réduite à 512 pixels au maximum.

In [ ]:
if SORTIE.exists():
    shutil.rmtree(SORTIE)  # on repart de zéro à chaque exécution

images["fichier"] = ""
compteur = {}
for i, ligne in images.iterrows():
    cle = (ligne["source"], ligne["classe"])
    compteur[cle] = compteur.get(cle, 0) + 1
    nom = f"{ligne['source']}_{ligne['classe']}_{compteur[cle]:04d}.jpg"
    destination = SORTIE / ligne["partie"] / ligne["classe"] / nom
    destination.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(SOURCES / ligne["origine"]) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.thumbnail((TAILLE_MAX, TAILLE_MAX), Image.Resampling.LANCZOS)
        im.save(destination, quality=92)
    images.at[i, "fichier"] = str(destination.relative_to(RACINE).as_posix())
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(images)} images copiées")

print(f"Terminé : {len(images)} images dans {SORTIE}")

## 5. Vérifications et manifeste

In [ ]:
parties_par_image = images.groupby("md5")["partie"].nunique()
print("Images présentes dans plusieurs parties :", int((parties_par_image > 1).sum()))

for partie in ["train", "val", "test"]:
    for classe in ["bon_etat", "endommage"]:
        n_disque = len(list((SORTIE / partie / classe).glob("*.jpg")))
        n_attendu = len(images[(images["partie"] == partie) & (images["classe"] == classe)])
        statut = "OK" if n_disque == n_attendu else "ÉCART"
        print(f"  {partie:<6} {classe:<10} {n_disque:>5} fichiers  {statut}")

MANIFESTE.parent.mkdir(exist_ok=True)
images[["fichier", "partie", "classe", "source", "groupe", "origine", "md5"]].to_csv(MANIFESTE, index=False, encoding="utf-8")
print(f"\nManifeste enregistré : {MANIFESTE}")

## 6. Aperçu du jeu préparé

Quatre images d'entraînement par source et par classe.

In [ ]:
random.seed(1)
groupes = list(images[images["partie"] == "train"].groupby(["source", "classe"]))
fig, axes = plt.subplots(len(groupes), 4, figsize=(10, 2.7 * len(groupes)), squeeze=False)
for i, ((source, classe), groupe) in enumerate(groupes):
    for j, fichier in enumerate(random.sample(list(groupe["fichier"]), min(4, len(groupe)))):
        axes[i, j].imshow(Image.open(RACINE / fichier))
        axes[i, j].axis("off")
    axes[i, 0].set_title(f"{source} – {classe.replace('_', ' ')}", loc="left", fontsize=10,
                         color="#2f9e44" if classe == "bon_etat" else "#c92a2a")
plt.tight_layout()
plt.show()